# Validation of flatcor's three default priors

This notebook presents the ten studies of the validation of [flatcor](https://github.com/dm-k1/flatcor) 1.0.0,
which compares three default priors for Bayesian linear regression: flat partial *r*, flat *R* and
flat *R*², each a Beta(*a*, 1) prior on *R*² with a uniform direction. The studies follow the
order of [`docs/appendix/C_study_design.md`](docs/appendix/C_study_design.md): correctness (1–4), formal properties (5–6) and behaviour (7–10).

Each study is a script in `R/studies/` that saves its tables and checks to
`results/study_NN.rds` and its figures to `figures/`; this notebook only
presents them. Every estimate carries its Monte Carlo standard error (MCSE), and every check is
judged relative to it.

**The setting below** chooses what the notebook does:

- `"results"` reads the saved results only (no computing; the default);
- `"cached"` reruns the studies first, reusing what is cached in `cache/`, so only what has
  changed is recomputed;
- `"fresh"` deletes each study's cache and reruns it from scratch.

The full validation takes hours (study 9 alone about six hours on seven cores); [`cluster/README.md`](cluster/README.md)
describes a run on a cluster. With `smoke <- TRUE` the notebook runs or reads the smoke test
instead: a corner of each design and 10 datasets, to test the code. Its results and figures go to
`cache/`, and its checks mean nothing.

In [ ]:
mode    <- "results"   # "results", "cached" or "fresh"
smoke   <- FALSE       # TRUE: the smoke test instead of the full studies
studies <- sprintf("%02d", 1:10)   # the studies to run and show
cores   <- NA          # cores per study when running; NA: all but one

In [ ]:
## Jupyter starts in the notebook's folder, the project folder, where the scripts run too
source("R/notebook/present.R")   # run_studies(), show_study(), show_table()
stopifnot(mode %in% c("results", "cached", "fresh"))
if (mode != "results") {
  runs <- run_studies(studies, fresh = mode == "fresh", smoke = smoke, cores = cores)
  show_table(runs, "This run (exit status 1: a check failed or the study stopped; see its log)")
}
show_table(overview_of(studies, smoke), if (smoke) "Overview (smoke test)" else "Overview")

---

# Correctness

## 1. Is the prior sampled correctly?

flatcor's Stan model samples *R*, with *R*² ~ Beta(*a*, 1), and a uniform direction, and computes
the coefficients from them. Sampling the prior only, does it reproduce each of the three priors?
Each setting is sampled until every checked quantity has enough effective draws for a
Kolmogorov–Smirnov check at the .01 level with a critical distance of at most .01.

In [ ]:
show_study("01", smoke = smoke, tables = list(
  "Sampling per setting" = function(t) data.frame(setting = names(t$iterations), iterations = t$iterations, smallest_ess = t$smallest_ess)
))

## 2. Are the Bayes factors computed correctly?

Every Bayes factor flatcor reports equals the integral it stands for: the whole-model Bayes factor
and the conditional test of one coefficient or of a block of columns. Each closed form is checked
against routes that share no code with the package: adaptive integration, a 200-bit ground truth
(Rmpfr), brute force from the prior, and bridge sampling on the package's Stan fit. The two Monte
Carlo routes run until their own error is below 0.01 on the log scale and are judged against it.

In [ ]:
show_study("02", smoke = smoke)

## 3. Is the posterior calibrated?

Simulation-based calibration (Talts et al., 2018) of the Stan model under each prior, with the SBC
package: with parameters and data drawn from the prior, the rank of each true value among its
posterior draws is uniform if the sampler is right. Flat *R* sampled directly in the coefficients
is shown as a contrast that is expected to fail (it is not a check).

In [ ]:
show_study("03", smoke = smoke, tables = list(
  "Rank uniformity (chi-square over 20 bins)" = "uniformity"
))

## 4. Is the Bayes factor calibrated?

On data drawn from each prior, the share of true nulls among datasets with a given BF₁₀ equals
the posterior probability of the null, 1 / (1 + BF₁₀), with equal prior odds (Schad et al., 2023).
On data drawn from another prior (JZS) it need not.

In [ ]:
show_study("04", smoke = smoke, tables = list(
  "Calibration per generating prior and test" = "calibration_tests",
  "Datasets each test calls null" = "called_null",
  "Observed and predicted share of nulls per bin of log BF10" = "calibration_bins"
))

---

# Formal properties

## 5. Does each prior meet the default-prior criteria?

The criteria of Bayarri, Berger, Forte and García-Donato (2012) for each prior: predictive
matching, information consistency, intrinsic consistency, and model-selection consistency for a
fixed and a growing number of predictors *k*. (Invariance is checked in study 2.)

In [ ]:
show_study("05", smoke = smoke, tables = list(
  "Rate of the log Bayes factor in log n" = "rate_slopes",
  "Median log Bayes factor under the alternative" = "alternative_medians",
  "Growing k: rate in log n" = "growing_slopes"
))

## 6. Is the test of one coefficient coherent with one model?

Does the conditional test of *x*₁ equal the Bayes factor of the full model against its restriction
to a zero partial correlation of *x*₁ (the Savage–Dickey ratio), and is that restriction a
proper model?

In [ ]:
show_study("06", smoke = smoke, tables = list(
  "Conditional test minus Savage-Dickey (estimable datasets)" = "coherence",
  "The restriction rho1 = 0, approached by |rho1| < eps" = "near_zero"
))

---

# Behaviour

## 7. How does the test of one coefficient behave?

How often each test supports the null (BF₀₁ > 3), supports an effect (BF₁₀ > 3) or stays
undecided about one coefficient, how that depends on the number of predictors *k*, and whether
differences between priors are differences in accuracy (how datasets are ranked, the AUC) or only
in where the test decides. All methods analyse the same datasets (set A), so every comparison is
paired.

In [ ]:
show_study("07", smoke = smoke, tables = list(
  "Share of each answer, pooled over the cells at each k" = function(t) answers_side_by_side(t$shares_per_k, c("k", "slope_x1", "method")),
  "Separating slope 0 from slope 0.1: AUC per design" = "auc"
))

## 8. How does it behave for factors?

The block conditional test (a term of *q* columns, a factor with *q* + 1 levels) against the joint
test and JZS: how often each supports the null, supports an effect or stays undecided, how well
each separates a null factor from a small one, and whether its evidence depends on what the other
predictors explain (it should not: the conditional Lindley paradox).

In [ ]:
show_study("08", smoke = smoke, tables = list(
  "Share of each answer, pooled over q" = function(t) answers_side_by_side(t$shares_over_q, c("n", "other_predictors", "factor_effect", "explained_by_others", "method")),
  "Separating a null factor from a small one: AUC" = "auc",
  "Slope of the median log BF10 for a null factor in what the others explain" = "paradox"
))

## 9. How good are the estimates?

Bias, interval coverage and prediction error of the Stan posterior under each prior, against
ordinary least squares and the BayesFactor posteriors (JZS, Zellner–Siow), on the datasets of
study 7.

In [ ]:
show_study("09", smoke = smoke, tables = list(
  "Bias and coverage of the slope of x1" = "bias_and_coverage",
  "Prediction error relative to OLS" = "prediction_ratio"
))

## 10. How does it select variables?

Variable selection over all 2¹² = 4,096 subsets of twelve candidate predictors, of which four
matter: how many real predictors each method misses and how many null ones it includes, and
whether methods differ in how well they rank real above null predictors (the AUC) or only in
where they cut.

In [ ]:
show_study("10", smoke = smoke, tables = list(
  "Per method and setting" = "summaries",
  "Ranking real above null predictors: AUC" = "auc"
))